# ContractRisk — Mistral con entrada nativa corregida

Sesión nueva de Colab con GPU T4. Es el mismo Mistral 7B, no un tercer modelo. Se mantiene checkpoint, rúbrica, veinte anclas aprobadas, seis pares, 38 juicios, semilla y parámetros. El cambio intencional es evitar añadir símbolos especiales extra después de aplicar la plantilla del chat.

Antes de cargar pesos se comprueban los 38 mensajes con el tokenizer real: reproducir la longitud anterior, detectar el inicio duplicado y verificar que la entrada corregida coincide exactamente con la plantilla nativa. La generación también verifica esa equivalencia en cada llamada.

Las puntuaciones anteriores se conservan. Corregir tokens no demuestra mejor juicio: el modelo debe superar los mismos criterios. Si el perfil GPU/paquetes registrado difiere del anterior, la comparación se detiene; no cambiar versiones manualmente.

Habilitar acceso de este notebook a WANDB_API_KEY y ejecutar en orden. No modifica main ni Space. Aunque supere la regresión, la validación final seguirá pendiente de confirmación separada y auditoría de razones.


In [ ]:
import os, sys, subprocess, tempfile, json
from pathlib import Path

ROOT = Path(tempfile.mkdtemp(prefix='contractrisk_final_tokenfix_', dir='/content')) / 'repo'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'Entrega/contractrisk-final',
    'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git', str(ROOT)], check=True)
os.chdir(ROOT)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Commit:', COMMIT)

def run_visible(arguments, logfile):
    with open(logfile, 'w', encoding='utf-8') as log:
        p = subprocess.Popen(arguments, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line); log.flush()
        status = p.wait()
    if status:
        raise RuntimeError(f'Proceso terminó con código {status}; consultar {logfile}')

from google.colab import userdata
try:
    key = userdata.get('WANDB_API_KEY')
    if not key:
        raise ValueError('Empty secret')
    os.environ['WANDB_API_KEY'] = key
    del key
except Exception:
    raise RuntimeError('Habilita el acceso de ESTE notebook a WANDB_API_KEY en el panel de secretos de Colab. No pegues ni imprimas la clave.') from None
print('Secreto accesible; su valor no se imprime. La conexión se validará antes de cargar modelos.')


In [ ]:
run_visible([sys.executable, '-m', 'pip', 'install', '-r',
    'final/requirements-calibration.txt'], 'installation.log')
import torch
assert torch.cuda.is_available(), 'Selecciona GPU T4 y una sesión nueva.'
print('GPU:', torch.cuda.get_device_name(0))
run_visible([sys.executable, '-c', "from pathlib import Path; from final.run_judge_calibration import verify_review; verify_review(Path('final/eval/calibration_review_v1')); print('20 anclas originales aprobadas; hashes y rúbrica comprobados.')"], 'review_check.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'final/tests', '-v'], 'tests_final.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests_m3.log')


In [ ]:
run_visible([sys.executable, '-u', 'final/tokenization_preflight.py', '--output', str(ROOT / 'tokenization_preflight.json')], 'tokenization_preflight.log')


In [ ]:
RESULTS = ROOT / 'mistral_tokenfix_results'
run_visible([sys.executable, '-u', 'final/run_mistral_tokenfix.py', '--output', str(RESULTS), '--require-wandb'], 'mistral_tokenfix.log')


## Inspección y evidencia

completed_regression_failed conserva el rechazo. completed_regression_passed_confirmation_pending todavía no valida al juez para la entrega. No reemplazar notas ni repetir para elegir mejores puntuaciones.

La descarga incluye el preflight de tokens, metadata, juicios originales y logs. También permite conservar una ejecución que falló; no contiene claves.


In [ ]:
RESULTS = ROOT / 'mistral_tokenfix_results'
for name in ['metadata.json', 'regression_summary.json']:
    path = RESULTS / name
    if path.exists():
        print(name, path.read_text(encoding='utf-8'))


In [ ]:
import shutil
from google.colab import files
RESULTS.mkdir(exist_ok=True)
(RESULTS / 'notebook_commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, RESULTS / log.name)
reviewed = RESULTS / 'reviewed_calibration_pack'
if not reviewed.exists():
    shutil.copytree(ROOT / 'final/eval/calibration_review_v1', reviewed)
shutil.copy2(ROOT / 'final/judge_second_config.json', RESULTS / 'judge_second_config.json')
proof = ROOT / 'tokenization_preflight.json'
if proof.exists():
    shutil.copy2(proof, RESULTS / proof.name)
package = shutil.make_archive(str(ROOT / 'contractrisk_final_mistral_tokenfix_resultados'), 'zip', RESULTS)
files.download(package)
